# Übung 02 – Modellbewertung bei unausgeglichenen Klassen: Online Shoppers

## Lernziel

Ich lerne, warum Accuracy bei seltenen positiven Ereignissen allein nicht genügt. Ich vergleiche eine triviale Baseline mit zwei Modellen, bewerte Precision, Recall, F1 und ROC-AUC und untersuche, wie die Entscheidungsschwelle die Art der Fehler verändert.

## Datensatz und Quelle

| Angabe | Quelle |
|---|---|
| Offizielle Dokumentation | https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset |
| Direkter Download | https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip |
| DOI | https://doi.org/10.24432/C5F88Q |
| Zitierform | Sakar, C. & Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset* [Dataset]. UCI Machine Learning Repository. |
| Lizenz laut UCI | CC BY 4.0 |

Die UCI-Dokumentation nennt 12.330 Sitzungen, von denen 84,5 % nicht mit einem Kauf enden. Genau deshalb ist der Fall geeignet, um die Metrikentscheidung kritisch zu üben.

Für die Einordnung von Precision-Recall-Diagrammen bei unausgeglichenen Klassen siehe Saito, T. & Rehmsmeier, M. (2015). *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432.

In [ ]:
# Ich halte alle Imports am Anfang zusammen. Dadurch ist transparent,
# welche Bibliotheken mein Notebook benötigt und ich kann Fehler schneller eingrenzen.
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Die feste Zufallszahl macht zufällige Aufteilungen und Modellresultate reproduzierbar.
SEED = 42
np.random.seed(SEED)

# Diese Darstellung ist für die Analyse in Colab gut lesbar.
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

DATA_DIR = Path('daten')
DATA_DIR.mkdir(exist_ok=True)


def download_and_extract_zip(url: str, label: str) -> Path:
    """Lädt ein offizielles ZIP-Archiv nur bei Bedarf herunter und entpackt es.

    Die Funktion ist absichtlich im Notebook sichtbar: Studierende sollen erkennen,
    dass die Datenquelle nicht manuell und nicht über einen lokalen Pfad bereitgestellt wird.
    """
    zip_path = DATA_DIR / f'{label}.zip'
    extract_dir = DATA_DIR / label

    if not zip_path.exists():
        print(f'Lade Daten von: {url}')
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        zip_path.write_bytes(response.content)
    else:
        print(f'Verwende vorhandenes Archiv: {zip_path}')

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True)
        with ZipFile(zip_path) as archive:
            archive.extractall(extract_dir)

    return extract_dir

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    precision_recall_curve, classification_report, f1_score, roc_auc_score,
    precision_score, recall_score, make_scorer
)

## 1. Daten laden und Zielvariable erzeugen

`Revenue` ist im Original ein Wahr/Falsch-Wert. Ich überführe ihn in 0 und 1, weil dies für viele Funktionen und Visualisierungen praktisch ist. Die Bedeutung bleibt transparent: 1 steht für eine Sitzung mit Kaufabschluss.

In [ ]:
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'
extract_dir = download_and_extract_zip(SOURCE_URL, 'online_shoppers')
csv_path = extract_dir / 'online_shoppers_intention.csv'
assert csv_path.exists(), f'Die erwartete Datei fehlt: {csv_path}'

df = pd.read_csv(csv_path)
df['Revenue'] = df['Revenue'].astype(bool).astype(int)
print(f'Datensatzform: {df.shape[0]:,} Zeilen und {df.shape[1]} Spalten')
df.head()

## 2. Warum Accuracy nicht genügt

Wenn ich immer „kein Kauf“ vorhersage, kann die Accuracy hier hoch sein, obwohl ich **keinen einzigen Kaufabschluss** erkenne. Diese triviale Regel wird als Dummy-Baseline bewusst mitgerechnet. Ein neues Modell muss besser sein als diese sehr einfache Referenz.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Revenue', color='#3b7ddd', ax=ax)
ax.set(title='Klassenverteilung: Kaufabschluss je Sitzung', xlabel='Revenue (0 = nein, 1 = ja)', ylabel='Anzahl Sitzungen')
plt.show()

share = df['Revenue'].value_counts(normalize=True).mul(100).round(2)
display(share.rename('Anteil in Prozent').to_frame())
print(f'Accuracy einer Regel, die immer 0 vorhersagt: {(df["Revenue"] == 0).mean():.2%}')

## 3. Datenaufteilung und Pipeline

Ich nutze dieselben guten Gewohnheiten wie im Bank-Fall: Split vor dem Fit, Stratifizierung und eine Pipeline. Der Datensatz enthält numerische und kategoriale Merkmale. `PageValues` ist ein fachlich interessanter, aber besonders kritisch zu diskutierender Prädiktor: Ich verwende ihn hier als Lehrbeispiel, dokumentiere aber ausdrücklich, dass seine zeitliche Verfügbarkeit im konkreten Einsatzkontext geprüft werden müsste.

In [ ]:
X = df.drop(columns='Revenue')
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(exclude='number').columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('numerisch', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('kategorial', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

print(f'Training: {len(X_train):,} | Test: {len(X_test):,}')
print('Numerische Merkmale:', numeric_features)
print('Kategoriale Merkmale:', categorical_features)

## 4. Baseline und Modelle in der Cross-Validation vergleichen

Die Dummy-Baseline sagt immer die häufigste Klasse voraus. Die logistische Regression liefert ein gut nachvollziehbares Modell. Der Random Forest bildet zusätzlich nichtlineare Muster ab. Ich vergleiche nicht nur Accuracy, sondern auch Recall, Precision, F1 und ROC-AUC.

In [ ]:
models = {
    'Dummy: häufigste Klasse': DummyClassifier(strategy='most_frequent'),
    'Logistische Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, class_weight='balanced', n_jobs=1, random_state=SEED
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Die Dummy-Baseline kann bewusst keine positiven Fälle vorhersagen. Ich setze zero_division=0,
# damit dieser fachlich aussagekräftige Fall sauber als Precision = 0 statt als Warnung erscheint.
scoring = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0)
}
rows = []

for name, model in models.items():
    pipeline = Pipeline(steps=[('vorbereitung', preprocessor), ('modell', model)])
    result = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({'Modell': name, **{metric: result[f'test_{metric}'].mean() for metric in scoring}})

cv_results = pd.DataFrame(rows).set_index('Modell').sort_values('f1', ascending=False)
display(cv_results.style.format('{:.3f}'))

## 5. Begrenztes Hyperparameter-Tuning ohne Testdatenkontakt

Ich tune hier nur die Regularisierungsstärke der logistischen Regression. Der Suchraum ist klein und transparent. Wichtig ist die Reihenfolge: Die Suche läuft nur auf Trainingsdaten und bewertet Kandidaten per Cross-Validation. Die Testmenge bleibt weiterhin unberührt.

In [ ]:
logistic_pipeline = Pipeline(steps=[
    ('vorbereitung', preprocessor),
    ('modell', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))
])

search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid={'modell__C': [0.1, 1.0, 10.0]},
    scoring='f1',
    cv=cv,
    n_jobs=-1
)
search.fit(X_train, y_train)

print('Beste Parameter:', search.best_params_)
print('Beste CV-F1:', round(search.best_score_, 3))

## 6. Schwellenwert bestimmen, ohne die Testmenge zu verwenden

Standardmäßig klassifiziert ein binäres Modell ab einer Wahrscheinlichkeit von 0,5 als positiv. Das ist keine Naturkonstante. Wenn ein Unternehmen nur sehr wenige Sessions nachbearbeiten kann, priorisiert es möglicherweise Precision. Wenn es möglichst keine kaufbereiten Personen übersehen möchte, priorisiert es eher Recall.

Methodisch entscheidend ist die Datenrolle: **Der Schwellenwert wird ausschließlich aus den Trainingsdaten bestimmt.** Dazu erzeuge ich Out-of-Fold-Wahrscheinlichkeiten. Jeder Trainingsfall erhält dadurch eine Prognose von einem Modell, das diesen Fall beim Fit nicht gesehen hat. Die Testmenge bleibt vollständig unberührt und wird erst nach der Schwellenwertentscheidung genau einmal ausgewertet.

In einer realen Anwendung könnte die Schwelle statt über maximales F1 auch direkt aus Fehlerkosten, Kapazitätsgrenzen oder einer verbindlichen Mindest-Precision abgeleitet werden.

In [ ]:
final_model = search.best_estimator_

# Out-of-Fold-Prognosen entstehen ausschließlich innerhalb der Trainingsdaten.
# Jeder Fall wird von einem Modell prognostiziert, das diesen Fall nicht zum Fit verwendet hat.
oof_proba = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

precision_oof, recall_oof, thresholds_oof = precision_recall_curve(y_train, oof_proba)
f1_oof = 2 * precision_oof * recall_oof / (precision_oof + recall_oof + 1e-12)
best_threshold_index = np.nanargmax(f1_oof[:-1])
best_threshold = float(thresholds_oof[best_threshold_index])

print('Beste Parameter:', search.best_params_)
print('Per Out-of-Fold-Prognosen gewählte Schwelle:', round(best_threshold, 3))
print('OOF-F1 an dieser Schwelle:', round(float(f1_oof[best_threshold_index]), 3))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds_oof, precision_oof[:-1], label='Precision')
ax.plot(thresholds_oof, recall_oof[:-1], label='Recall')
ax.plot(thresholds_oof, f1_oof[:-1], label='F1')
ax.axvline(best_threshold, color='black', linestyle='--', label=f'gewählte Schwelle = {best_threshold:.2f}')
ax.set(
    title='Schwellenwertwahl mit Out-of-Fold-Trainingsprognosen',
    xlabel='Entscheidungsschwelle',
    ylabel='Kennzahl',
    ylim=(0, 1)
)
ax.legend()
plt.show()

# Erst nach der Festlegung der Schwelle wird das bereits auf allen Trainingsdaten
# gefittete beste Modell genau einmal auf der Testmenge ausgewertet.
y_proba = final_model.predict_proba(X_test)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)
y_pred_selected = (y_proba >= best_threshold).astype(int)

print('Test-ROC-AUC:', round(roc_auc_score(y_test, y_proba), 3))
print('Test-F1 bei Schwelle 0,50:', round(f1_score(y_test, y_pred_default), 3))
print(f'Test-F1 bei vorab gewählter Schwelle {best_threshold:.2f}:', round(f1_score(y_test, y_pred_selected), 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_selected,
    display_labels=['kein Kauf', 'Kauf'],
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix: feste Schwelle {best_threshold:.2f}')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('ROC-Kurve auf unberührten Testdaten')

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2])
axes[2].set_title('Precision-Recall auf unberührten Testdaten')
plt.tight_layout()
plt.show()

print(f'\nBericht bei vorab gewählter Schwelle {best_threshold:.2f}:')
print(classification_report(y_test, y_pred_selected, target_names=['kein Kauf', 'Kauf']))

## 7. Transferfragen

1. Warum ist die Dummy-Baseline hier methodisch notwendig?
2. Welche Metrik würden Sie wählen, wenn nur 50 Sitzungen pro Tag nachbearbeitet werden können?
3. Wie verändert eine niedrigere Schwelle typischerweise Precision und Recall?
4. Warum wäre es methodisch falsch, die optimale Schwelle anhand der finalen Testlabels zu bestimmen?
5. Wie könnte eine Schwelle statt über maximales F1 über konkrete Fehlerkosten oder eine Kapazitätsgrenze festgelegt werden?

> **Merksatz:** Metriken und Schwellenwerte übersetzen Vorhersagen in Entscheidungen. Modellwahl, Schwellenwertwahl und finale Testbewertung benötigen klar getrennte Datenrollen.